In [1]:
!pip install -q "Pillow>=10.2.0,<12.0"
!pip install -q transformers>=4.40 "bitsandbytes>=0.46.1" "accelerate>=0.25" datasets matplotlib numpy pandas tqdm scipy

In [5]:
import os
import sys
from pathlib import Path

# Thesis root is the parent of the medgemma/ notebook directory.
_thesis_root = str(Path(os.getcwd()).parent)
if _thesis_root not in sys.path:
    sys.path.insert(0, _thesis_root)
print('Thesis root:', _thesis_root)
print(os.listdir(_thesis_root))


['__pycache__', 'aggregate_perturbation.png', 'all_results.json', 'generate_technical_report.py', 'llava_med', 'removed_module.py', 'requirements.txt', 'technical_report.pdf', 'test.py', 'saliency', 'test_images', '.git', 'model_utils.py', 'run.py', 'evaluation.py', 'analyze_results.py', '.venv', 'NER_LLAVA.png', 'llavamedtime.txt', 'config.json', 'llava_med_colab_pipeline', '.gitignore', 'config.py', 'dataset.py', 'gemma', 'explainability_medgemma_3config_study.ipynb', 'medgemma_layer_study', 'final', 'final-content', 'visualization.py', 'explainability_medgemma.ipynb']


In [6]:
from medgemma.config import Config
from pathlib import Path

# -- Dataset toggle -------------------------------------------------------------
USE_COCO = False  # True -> COCO (MS-COCO captions) | False -> ROCO v2

cfg = Config()
cfg.model_id = 'google/medgemma-1.5-4b-it'
cfg.load_in_4bit = True
cfg.attn_implementation = 'eager'

# Results are stored under <thesis_root>/final-content/
base_results_dir = Path.cwd().parent / 'final-content'

if USE_COCO:
    # lmms-lab/COCO-Caption: image column already contains PIL images.
    cfg.dataset_name = 'lmms-lab/COCO-Caption'
    cfg.dataset_config = ''
    cfg.dataset_split = 'val'
    cfg.image_column = 'image'
    cfg.caption_column = 'answer'   # list[str]; loader uses the first caption
    cfg.prompt = "Generate a caption for this image."
    cfg.output_dir = str(base_results_dir / 'results_medcocoh')
else:
    cfg.dataset_name = 'eltorio/ROCOv2-radiology'
    cfg.dataset_config = ''
    cfg.dataset_split = 'train'
    cfg.image_column = 'image'
    cfg.caption_column = 'caption'
    cfg.prompt = "Write a single sentence caption for this image."
    cfg.output_dir = str(base_results_dir / 'results_medroco_raw')

cfg.num_samples = 100
cfg.max_new_tokens = 100
cfg.methods = ['gradcam', 'attention', 'gmar_l1', 'gmar_l2']
cfg.mask_ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
cfg.save_visualizations = True

EVAL_MODE = 'per_token'
CONTENT_ONLY = True

# -- Experiment toggles --------------------------------------------------------
# RUN_DELETION: runs the standard deletion (masking) faithfulness experiment.
#   Saves per-token probability drops to per_token_drops.* files.
RUN_DELETION = False

# RUN_INSERTION: runs an insertion experiment after the deletion experiment for
#   every method. Starts from an all-black image and progressively reveals the
#   top-k patches. Saves per-token probability data to insertion_per_token_drops.*
#   No saliency images are saved for the insertion experiment.
RUN_INSERTION = True

print(f"[config] Dataset: {'COCO (lmms-lab/COCO-Caption val)' if USE_COCO else 'ROCO v2 (eltorio/ROCOv2-radiology)'}")
print(f"[config] Output dir: {cfg.output_dir}")
print(f"[config] Deletion experiment: {RUN_DELETION}")
print(f"[config] Insertion experiment: {RUN_INSERTION}")


[config] Dataset: ROCO v2 (eltorio/ROCOv2-radiology)
[config] Output dir: /content/drive/Othercomputers/My Mac/Thesis/final-content/results_medroco_raw


In [7]:
from medgemma.model_utils import load_model_and_processor
model, processor = load_model_and_processor(cfg)

[model] Loading google/medgemma-1.5-4b-it …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

[model] Loaded.  device_map = n/a


In [9]:
from medgemma.dataset import load_dataset_samples
samples = load_dataset_samples(cfg)

[dataset] Loading 'eltorio/ROCOv2-radiology' split='train' (streaming, 250 samples) …


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Loading samples: 100%|██████████| 250/250 [00:02<00:00, 123.72it/s]

[dataset] Loaded 250 samples.


In [10]:
import importlib
import shutil
from pathlib import Path

import medgemma.model_utils as model_utils
import medgemma.evaluation as evaluation
import medgemma.saliency.gmar_shared
import medgemma.saliency.gmar_l1
import medgemma.saliency.gmar_l2
import medgemma.saliency.attention
import medgemma.saliency.gradcam
import medgemma.visualization as visualization

for m in [
    model_utils,
    evaluation,
    medgemma.saliency.gmar_shared,
    medgemma.saliency.gmar_l1,
    medgemma.saliency.gmar_l2,
    medgemma.saliency.attention,
    medgemma.saliency.gradcam,
    visualization
]:
    importlib.reload(m)

In [12]:
import gc
import json
import time
import torch
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm as tqdm_nb

from medgemma.model_utils import (
    generate_caption, get_tokenizer, get_image_token_positions,
    get_token_probabilities, get_content_token_mask,
    build_tf_inputs, move_inputs_to_device,
)
from medgemma.saliency import get_saliency_fn
from medgemma.evaluation import (
    evaluate_faithfulness_average,
    evaluate_faithfulness_per_token,
    evaluate_insertion_per_token,
)
from medgemma.visualization import (
    save_token_saliency_grid, save_comparison_figure,
    save_perturbation_curve, save_aggregate_curves,
)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'config.json', 'w') as f:
    json.dump(vars(cfg), f, indent=2, default=str)

tok = get_tokenizer(processor)
all_sample_results = []

# -----------------------------------------------------------------------------
# Main loop
# -----------------------------------------------------------------------------
t_total = time.time()

for i in tqdm_nb(range(len(samples)), desc='Processing samples'):
    sample = samples[i]
    image = sample['image']
    ref_caption = sample.get('caption', '')
    sample_id = sample.get('id', str(i))

    # 1. Generate caption
    gen_ids, gen_text, input_len, inputs = generate_caption(model, processor, image, cfg)
    total_len = gen_ids.shape[1]
    num_gen = total_len - input_len
    print(f'\nSample {i}: generated {num_gen} tokens - {gen_text[:100]}')
    if num_gen == 0:
        continue

    # 2. Image-token positions
    img_positions = get_image_token_positions(inputs)

    # 3. Teacher-forcing inputs
    tf_inputs = build_tf_inputs(inputs, gen_ids, input_len)

    # 4. Original token probabilities (needed for deletion; skip if deletion off)
    orig_probs = get_token_probabilities(model, tf_inputs, gen_ids, input_len) if RUN_DELETION else None

    # 5. Token strings & content mask
    token_strings = {}
    for pos in range(input_len, total_len):
        tid = gen_ids[0, pos].item()
        token_strings[pos] = tok.decode([tid], skip_special_tokens=True).strip()
    content_mask = get_content_token_mask(tok, gen_ids, input_len)
    content_positions = [
        pos for pos, keep in zip(range(input_len, total_len), content_mask) if keep
    ]

    # 6. Saliency + evaluation per method
    sample_dir = out_dir / f'sample_{i:04d}'
    sample_dir.mkdir(parents=True, exist_ok=True)

    all_saliency        = {}
    method_eval_results = {}
    method_ins_results  = {}

    token_positions = list(range(input_len, total_len))
    gen_text_str = gen_text

    for method in cfg.methods:
        print(f'  {method} ...')
        compute  = get_saliency_fn(method)
        sal_maps = compute(model, tf_inputs, gen_ids, input_len, img_positions, cfg)
        all_saliency[method] = sal_maps

        # ── Deletion evaluation ────────────────────────────────────────
        if RUN_DELETION:
            var_content_mask = [
                content_mask[pos - input_len] if CONTENT_ONLY else True
                for pos in token_positions
            ]
            ev = evaluate_faithfulness_per_token(
                model, inputs, gen_ids, input_len, sal_maps, orig_probs, cfg,
                content_mask=var_content_mask,
            )
            for row in ev.get('per_token', []):
                row['token_text'] = token_strings.get(row.get('position'), '')
            print(f'  [del] AOPC = {ev["aopc"]:.4f}')
            method_eval_results[method] = ev

        # ── Insertion evaluation ───────────────────────────────────────
        if RUN_INSERTION:
            ins_content_mask = [
                content_mask[pos - input_len] if CONTENT_ONLY else True
                for pos in token_positions
            ]
            ins_ev = evaluate_insertion_per_token(
                model, inputs, gen_ids, input_len, sal_maps, cfg,
                content_mask=ins_content_mask,
            )
            for row in ins_ev.get('per_token', []):
                row['token_text'] = token_strings.get(row.get('position'), '')
            print(f'  [ins] AOPC_ins = {ins_ev["aopc_ins"]:.4f}')
            method_ins_results[method] = ins_ev

        if cfg.save_visualizations:
            save_token_saliency_grid(
                image, sal_maps, token_strings, method,
                str(sample_dir / f'saliency_{method}.png'),
                content_positions=content_positions if content_positions else None,
            )

    # 7. Random baseline
    random_positions = content_positions if CONTENT_ONLY else token_positions
    random_sal_maps = {
        pos: np.random.rand(cfg.image_token_grid, cfg.image_token_grid).astype(np.float32)
        for pos in random_positions
    }
    rand_content_mask = [
        pos in random_positions for pos in token_positions
    ]

    if RUN_DELETION:
        rand_ev = evaluate_faithfulness_per_token(
            model, inputs, gen_ids, input_len, random_sal_maps, orig_probs, cfg,
            content_mask=rand_content_mask,
        )
        for row in rand_ev.get('per_token', []):
            row['token_text'] = token_strings.get(row.get('position'), '')
        print(f'  [random del] AOPC = {rand_ev["aopc"]:.4f}')
        method_eval_results['random'] = rand_ev

    if RUN_INSERTION:
        rand_ins_ev = evaluate_insertion_per_token(
            model, inputs, gen_ids, input_len, random_sal_maps, cfg,
            content_mask=rand_content_mask,
        )
        for row in rand_ins_ev.get('per_token', []):
            row['token_text'] = token_strings.get(row.get('position'), '')
        print(f'  [random ins] AOPC = {rand_ins_ev["aopc_ins"]:.4f}')
        method_ins_results['random'] = rand_ins_ev

    all_saliency['random'] = random_sal_maps

    # 8. Comparison & curve figures
    if cfg.save_visualizations:
        if len(all_saliency) >= 2:
            save_comparison_figure(
                image, all_saliency, token_strings,
                str(sample_dir / 'comparison.png'),
                content_positions=content_positions if content_positions else None,
            )
        if method_eval_results:
            save_perturbation_curve(
                method_eval_results, str(sample_dir / 'perturbation_curve.png'),
                title=f'Sample {i}',
            )
    image.save(str(sample_dir / 'original.png'))

    # 9. Collect result
    res = {
        'sample_id':            sample_id,
        'sample_idx':           i,
        'generated_text':       gen_text,
        'reference_caption':    ref_caption,
        'num_generated_tokens': num_gen,
        'num_content_tokens':   sum(content_mask),
        'eval': {
            m: {
                'aopc':                r.get('aopc', 0.0),
                'mean_drops_by_ratio': r.get('mean_drops_by_ratio', {}),
                'per_token':           r.get('per_token', []),
            }
            for m, r in method_eval_results.items()
        } if RUN_DELETION else {},
        'eval_insertion': {
            m: {
                'aopc_ins':            r.get('aopc_ins', 0.0),
                'mean_rises_by_ratio': r.get('mean_rises_by_ratio', {}),
                'per_token':           r.get('per_token', []),
            }
            for m, r in method_ins_results.items()
        } if RUN_INSERTION else {},
    }
    all_sample_results.append(res)

    with open(sample_dir / 'result.json', 'w') as f:
        json.dump(res, f, indent=2, default=str)

    # Free memory
    del tf_inputs, sal_maps
    if RUN_DELETION:
        del orig_probs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# -----------------------------------------------------------------------------
# Save final outputs
# -----------------------------------------------------------------------------
with open(out_dir / 'all_results.json', 'w') as f:
    json.dump(all_sample_results, f, indent=2, default=str)

# Per-token deletion rows
del_rows = [
    {**{'sample_idx': r['sample_idx'], 'sample_id': r['sample_id'], 'method': m}, **tok_row}
    for r in all_sample_results
    for m, ev in r.get('eval', {}).items()
    for tok_row in ev.get('per_token', [])
]
if del_rows:
    pd.DataFrame(del_rows).to_csv(out_dir / 'per_token_drops.csv', index=False)
    with open(out_dir / 'per_token_drops.jsonl', 'w') as f:
        for row in del_rows:
            f.write(json.dumps(row, default=str) + '\n')

# Per-token insertion rows
ins_rows = [
    {**{'sample_idx': r['sample_idx'], 'sample_id': r['sample_id'], 'method': m}, **tok_row}
    for r in all_sample_results
    for m, ev in r.get('eval_insertion', {}).items()
    for tok_row in ev.get('per_token', [])
]
if ins_rows:
    pd.DataFrame(ins_rows).to_csv(out_dir / 'insertion_per_token_drops.csv', index=False)
    with open(out_dir / 'insertion_per_token_drops.jsonl', 'w') as f:
        for row in ins_rows:
            f.write(json.dumps(row, default=str) + '\n')

# Summary
summary_rows = []
all_eval_by_method = {}
for res in all_sample_results:
    for m, ev in res.get('eval', {}).items():
        all_eval_by_method.setdefault(m, []).append(ev)
for method, evals in all_eval_by_method.items():
    aopc_vals = [e.get('aopc', 0.0) for e in evals]
    summary_rows.append({'method': method, 'mean_aopc': float(np.mean(aopc_vals)), 'std_aopc': float(np.std(aopc_vals)), 'n_samples': len(aopc_vals)})
if summary_rows:
    pd.DataFrame(summary_rows).to_csv(out_dir / 'summary.csv', index=False)

if cfg.save_visualizations and all_eval_by_method:
    save_aggregate_curves(all_eval_by_method, str(out_dir / 'aggregate_perturbation.png'))

elapsed = time.time() - t_total
print(f'\n{"="*60}')
print(f'Done. {len(all_sample_results)} samples in {elapsed:.0f}s')
print(f'Output: {out_dir}')
print(f'{"="*60}')


[resume] existing rows loaded: 218
[resume] start_idx=218 | end_idx=250 | total_samples=250
[checkpoint:start] samples=218 | output=/content/drive/Othercomputers/My Mac/Thesis/final-content/results_medroco_raw
[checkpoint:start] per-token rows=148975


Processing samples:   0%|          | 0/32 [00:00<?, ?it/s]


Sample 218: generated 19 tokens - CT scan of the chest showing a large mass in the left lung, potentially representing a tumor.
  gradcam ...


    [original] AOPC = 0.0365
  attention ...


    [original] AOPC = 0.0286
  gmar_l1 ...


...

    [original] AOPC = 0.0880
  attention ...


    [original] AOPC = 0.0693
  gmar_l1 ...


    [original] AOPC = 0.0632
  gmar_l2 ...


    [original] AOPC = 0.0628


: 

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Keep this cell aligned with the main run cell output location.
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
for method, evals in all_eval_by_method.items():
    if not evals:
        continue
    aopc_vals = [e.get('aopc', 0.0) for e in evals if isinstance(e, dict)]
    mean_aopc = float(np.mean(aopc_vals))
    std_aopc = float(np.std(aopc_vals))
    print(f'{method:>12s}:  AOPC = {mean_aopc:.4f} ± {std_aopc:.4f}')
    summary_rows.append({
        'method': method,
        'mean_aopc': mean_aopc,
        'std_aopc': std_aopc,
        'n_samples': len(aopc_vals),
    })

if summary_rows:
    df = pd.DataFrame(summary_rows)
    df.to_csv(out_dir / 'summary.csv', index=False)
    display(df)

with open(out_dir / 'all_results.json', 'w') as f:
    json.dump(all_sample_results, f, indent=2, default=str)

if cfg.save_visualizations and any(all_eval_by_method.values()):
    save_aggregate_curves(all_eval_by_method, str(out_dir / 'aggregate_perturbation.png'))

print('Results saved to', out_dir)

     gradcam:  AOPC = 0.0992 ± 0.0610
   attention:  AOPC = 0.1074 ± 0.0806
     gmar_l1:  AOPC = 0.1022 ± 0.0803
     gmar_l2:  AOPC = 0.1046 ± 0.0851


,method,mean_aopc,std_aopc,n_samples
0,gradcam,0.099197,0.060964,15
1,attention,0.107389,0.080644,15
2,gmar_l1,0.102157,0.080322,15
3,gmar_l2,0.104565,0.085131,15


Results saved to /content/drive/Othercomputers/My Mac/Thesis/results_medroco


In [ ]:
import shutil
from pathlib import Path

out_dir = Path(cfg.output_dir)
zip_path = out_dir.parent / f'{out_dir.name}_archive'
shutil.make_archive(str(zip_path), 'zip', str(out_dir))
print(f'Archive created: {zip_path}.zip  (source: {out_dir})')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered: /content/results_archive.zip
